In [1]:
import pandas as pd
from utils.utils import load_all_games_csv, get_teams, basic_win_prob_for_et
from elos.elo_tracker import EloTracker
from sklearn.metrics import log_loss, accuracy_score
from sklearn.linear_model import LogisticRegression
from scipy.special import expit
from matplotlib import pyplot as plt
from typing import Tuple
import numpy as np
from tqdm import tqdm


# Win Probability Analysis

This notebook will compare several different methods to estimate win probabilities from Elo ratings, and possibly home advantage, travel distance, and rest days.

## Get all Games

In [2]:
all_games = load_all_games_csv('../data/gameinfo_cleaned.csv')
all_games.head()

/Users/lancehendricks/Documents/College Coding/ML/Elo Ratings/analysis/src/utils/utils.py:27: DtypeWarning: Columns (10,11,13,17,19,21,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  all_games = pd.read_csv(filename)


,visteam,hometeam,site,date,number,starttime,daynight,innings,tiebreaker,usedh,...,homedistancetraveled,visdistancetraveled,homerestdays,visrestdays,homepitcherrgs,vispitcherrgs,hometeamrgs,visteamrgs,homepitcherminusteamrgs,vispitcherminusteamrgs
gid,,,,,,,,,,,,,,,,,,,,,
LS3189904140,CHN,LS3,LOU03,18990414,0.0,0:00PM,day,NaN,NaN,False,...,0.0,269.187008,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0
PHI189904140,WSN,PHI,PHI09,18990414,0.0,0:00PM,day,NaN,NaN,False,...,0.0,124.167063,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0
BLN189904150,NY1,BLN,BAL07,18990415,0.0,0:00PM,day,NaN,NaN,False,...,0.0,175.916531,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0
BRO189904150,BSN,BRO,NYC12,18990415,0.0,0:00PM,day,NaN,NaN,False,...,0.0,188.873979,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0
CIN189904150,PIT,CIN,CIN05,18990415,0.0,0:00PM,day,NaN,NaN,False,...,0.0,257.190815,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0


In [3]:
# Max rest days
all_games['visrestdays'] = all_games['visrestdays'].apply(lambda x: min(3,x))
all_games['homerestdays'] = all_games['homerestdays'].apply(lambda x: min(3,x))


In [4]:
# Take cube root of distance traveled
all_games['homedistancetraveled'] = all_games['homedistancetraveled']**(1/3)
all_games['visdistancetraveled'] = all_games['visdistancetraveled']**(1/3)

## Function for Evaluating Performance

In [5]:
def add_elos_to_games_df(games_df: pd.DataFrame, elo_prob_func=basic_win_prob_for_et, K: float = 3) -> pd.DataFrame:
    """Returns a version of games_df with columns 'homeelo' and 'viselo' added, which
    are calculated in part with the Elo probability function, elo_prob_func.
    
    Removes any na rows for important features at the end.
    
    Args:
        games_df (pd.DataFrame): Table whose rows are chronologically ordered game box scores,
                including columns 'hometeam' for the home team, 'visteam' for the away team, and
                'homewon' which is True if home won and False otherwise. Each game in game_df must take
                place after the games that have already been logged for the given teams it includes.
                Must be indexed by a game id column 'gid'.
        elo_prob_func (function): Function that takes in a home elo, away elo, and game information
            (i.e. row of box scores dataframe) and produces the probability of the home team winning.
        K (float): The K factor, controlling how sensitive each Elo update should be.
    """
    games_df = games_df.copy() # Don't modify original
    
    teams = get_teams(games_df)
    
    # First, get all Elo ratings
    et = EloTracker(teams, elo_prob_func=elo_prob_func, K=K)
    
    et.add_history(games_df)
    
    # Add raw pre-game Elo Ratings
    games_df['homeelo'] = [0.0] * len(games_df)
    games_df['viselo'] = [0.0] * len(games_df)
    
    home_elos = {}
    vis_elos = {}

    for team in teams:
        #print(len(et.elos_map[team]))
        for game in et.elos_map[team]:
            gid = game[0]
            elo = game[2] # Before update
            #print(elo)
        
            if games_df.loc[gid,'hometeam'] == team:
                home_elos[gid] = elo
            else:
                vis_elos[gid] = elo
                
    games_df['homeelo'] = games_df.index.map(home_elos)
    games_df['viselo'] = games_df.index.map(vis_elos)
                
    # Drop rows with na travel distance, rest or pitcher info
    games_df = games_df.dropna(subset=['homedistancetraveled', 'visdistancetraveled', 'homerestdays', 'visrestdays', 'homepitcherminusteamrgs', 'vispitcherminusteamrgs']).copy()
    
    #print(games_df[games_df['homeelo'].isna() | games_df['viselo'].isna()])
                
    return games_df

In [6]:
def evaluate_elo_prob_func(games_df: pd.DataFrame, elo_prob_func=basic_win_prob_for_et, K: float = 3) -> Tuple[float, float]:
    """Evaluates how well the given function to calculate Elo probabilties does on games_df,
    producing binary cross entropy and accuracy.
    
    Args:
        games_df (pd.DataFrame): Table whose rows are chronologically ordered game box scores,
                including columns 'hometeam' for the home team, 'visteam' for the away team, and
                'homewon' which is True if home won and False otherwise. Each game in game_df must take
                place after the games that have already been logged for the given teams it includes.
                Must be indexed by a game id column 'gid'.
        elo_prob_func (function): Function that takes in a home elo, away elo, and game information
            (i.e. row of box scores dataframe) and produces the probability of the home team winning.
        K (float): The K factor, controlling how sensitive each Elo update should be.
    """
    
    # Add elos
    games_df = add_elos_to_games_df(games_df, elo_prob_func, K=K)
        
    games_df['homewinprob'] = games_df.apply(lambda game: elo_prob_func(game['homeelo'], game['viselo'], game), axis=1)
    bce = log_loss(games_df['homewon'], games_df['homewinprob'])
    accuracy = accuracy_score(games_df['homewon'], round(games_df['homewinprob']))
    
    return bce, accuracy

## Evaluate Simple Probability model

In [7]:
bce, accuracy = evaluate_elo_prob_func(all_games, basic_win_prob_for_et)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6817783615812832
Accuracy: 0.5611052812190719


## With +28 Adjustment for Home Team

In [8]:
bce, accuracy = evaluate_elo_prob_func(all_games, lambda home_elo, away_elo, game_info: basic_win_prob_for_et(home_elo + 28, away_elo, game_info))
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6787546395806596
Accuracy: 0.5682569838514754


## With + 1.9% Adjustment for Home Team

In [9]:
bce, accuracy = evaluate_elo_prob_func(all_games, lambda home_elo, away_elo, game_info: basic_win_prob_for_et(home_elo*1.019, away_elo, game_info))
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6787388203430803
Accuracy: 0.5682669305867082


## With Logistic Regression

In [10]:
# First need to fit using some Elos - use the initial basic probability func.

df_to_fit = add_elos_to_games_df(all_games)


In [11]:
df_to_fit['elodiff'] = df_to_fit['viselo'] - df_to_fit['homeelo']
df_to_fit['restdiff'] = df_to_fit['visrestdays'] - df_to_fit['homerestdays']
df_to_fit['distancediff'] = df_to_fit['visdistancetraveled'] - df_to_fit['homedistancetraveled']
df_to_fit['homediff'] =  0 - 1
df_to_fit['pitcherdiff'] = df_to_fit['vispitcherminusteamrgs'] - df_to_fit['homepitcherminusteamrgs']

#features = ['elodiff', 'distancediff', 'restdiff']
features = ['elodiff', 'homediff', 'restdiff', 'distancediff', 'pitcherdiff']
X = df_to_fit[features].to_numpy()
y = df_to_fit['homewon'].astype(int).to_numpy().reshape(-1,1)

s = -np.log(10) / 400

In [12]:
# Fit via GD
w = np.zeros((5,1))
w[0,0] = s # Becomes 1 once dividing by s

step = 0.01 # Slightly higher for small gradients
iterations = 10000

for _ in range(iterations):

    z = X @ w
    y_hat = expit(z)
    
    w_grad = (1/X.shape[0]) * X.T @ (y_hat - y)
    
    w_grad[0,0] = 0
    
    #print(w_grad)
    
    w = w - step*w_grad
    
w

array([[-0.00575646],
       [-0.15456232],
       [-0.023462  ],
       [ 0.00160778],
       [-0.00839107]])

In [13]:
# Convert back to interpretable coefficients for individual Elo adjustments
w = (1/s) * w
w

array([[ 1.        ],
       [26.85022567],
       [ 4.07576704],
       [-0.27930024],
       [ 1.45767804]])

In [14]:
def p(X,w):
    """Vector form Elo pdf for a tabular input."""
    z = X @ w
    return expit((-np.log(10) / 400) * z)

In [15]:
def predict_lr(home_elo, away_elo, game):
    
    elo_diff = away_elo - home_elo
    rest_day_diff = game['visrestdays'] - game['homerestdays']
    rest_day_diff = rest_day_diff if not np.isnan(rest_day_diff) else 0
    
    travel_diff = game['visdistancetraveled'] - game['homedistancetraveled']
    travel_diff = travel_diff if not np.isnan(travel_diff) else 0
    
    home_adv_diff = 0 - 1
    
    pitcher_diff = game['vispitcherminusteamrgs'] - game['homepitcherminusteamrgs']
    pitcher_diff = pitcher_diff if not np.isnan(pitcher_diff) else 0
    
    
    x = np.array([elo_diff, home_adv_diff, rest_day_diff, travel_diff, pitcher_diff]).reshape(-1,1)
    
    if np.isnan(np.min(x)):
        print(x)
     
    return p(x.T, w).item()

In [16]:
bce, accuracy = evaluate_elo_prob_func(all_games, predict_lr)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6777509062033406
Accuracy: 0.571196244112776
